# WDL — Module 1 · Session 3 — Reading a rain gauge record; time scales
### Practice activity

| | |
|---|---|
| **Course** | Hydroinformatics in a Changing Climate: Water Data Literacy for All |
| **Department** | Civil Engineering · CHRIST (Deemed to be University) |
| **Instructor** | Dr Arpan Pradhan |
| **Module** | Module 1 — The Water Cycle as Data |
| **Session** | Session 3 of 30 |
| **CO** | CO1 |
| **Topics** | Rain gauges and the IMD rainfall day; rainy-day definition (2.5 mm); daily, weekly, monthly and seasonal totals; IMD seasons |
| **Water hook** | *IMD's 8:30 am rainfall day* |

---
> **How to use this notebook.** No coding needed. Read each explanation, then press ▶ on the box below it
> (or click the box and press `Shift + Enter`). Some boxes have sliders, number boxes or dropdowns: change them, then press ▶ again.
> Go to **File → Save a copy in Drive** first, so the notebook is yours.

Answer five questions about the sample station's 2023 record. Use the explorer below to find each answer, then check it.

In [ ]:
#@title ▶ Step 0 · Get ready (press ▶ once before anything else)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from IPython.display import display, Markdown
plt.rcParams.update({"figure.figsize": (11, 4.5), "figure.dpi": 110, "font.size": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, RED, GREY, TEAL = "#2C6E9B", "#DD8452", "#C44E52", "#8C8C8C", "#1F7A8C"
def ok(msg="Correct!"):  display(Markdown(f"✅ **{msg}**"))
def no(msg="Not quite. Check your working and try again."): display(Markdown(f"❌ {msg}"))
def close(a, b, tol=0.02):  # within 2% (or 0.01 for very small numbers)
    try: return abs(float(a) - float(b)) <= max(abs(float(b)) * tol, 0.01)
    except (TypeError, ValueError): return False

DATA_URL = "https://raw.githubusercontent.com/PradhanArpan/waterliteracy/main/data/M1/sample_station_2023.csv"
def _make_sample_station():
    rng = np.random.default_rng(2023)
    days = pd.date_range("2023-01-01", "2023-12-31", freq="D")
    p_wet  = [0.03, 0.03, 0.08, 0.20, 0.35, 0.40, 0.45, 0.48, 0.55, 0.50, 0.30, 0.08]
    mean_r = [3, 3, 6, 9, 10, 7, 7, 8, 11, 11, 9, 4]
    rain = []
    for d in days:
        m = d.month - 1
        rain.append(rng.gamma(0.8, mean_r[m] / 0.8) if rng.random() < p_wet[m] else 0.0)
    rain = np.round(np.array(rain), 1)
    rain[days.get_loc(pd.Timestamp("2023-09-21"))] = 96.4
    rain[days.get_loc(pd.Timestamp("2023-09-22"))] = 58.2
    level, soil, channel = [], 0.0, 0.0
    for r in rain:
        soil = soil * 0.6 + max(r - 5, 0) * 0.5
        channel = channel * 0.82 + soil * 0.022
        level.append(round(1.8 + channel, 2))
    return pd.DataFrame({"date": days, "rain_mm": rain, "level_m": level})
try:
    station = pd.read_csv(DATA_URL, parse_dates=["date"])
except Exception:
    station = _make_sample_station()   # works offline too: same numbers
station = station.set_index("date")
rainfall, river = station[["rain_mm"]], station[["level_m"]]
WARNING_LEVEL, DANGER_LEVEL = 3.6, 4.2
display(Markdown("✅ **Ready.** Sample station loaded: one year (2023) of daily rainfall and river level. "
                 "*Illustrative, synthetic data for practice; not an official IMD or CWC record.*"))

---
## Explorer
Choose what to show, press ▶, and read off the numbers you need.

In [ ]:
#@title ▶ Rainfall explorer { run: "auto" }
show = "Monthly totals" #@param ["Monthly totals", "Seasonal totals", "Ten wettest days", "Days at or above a threshold"]
threshold_mm = 2.5 #@param {type:"number"}
r = rainfall.rain_mm
if show == "Monthly totals":
    t = r.resample("MS").sum().round(1); t.index = t.index.strftime("%b"); display(t.to_frame("Rain (mm)").T)
elif show == "Seasonal totals":
    sea = r.index.month.map(lambda m: "Winter (Jan–Feb)" if m <= 2 else "Pre-monsoon (Mar–May)" if m <= 5 else "SW monsoon (Jun–Sep)" if m <= 9 else "NE monsoon (Oct–Dec)")
    display(r.groupby(sea).sum().round(1).to_frame("Rain (mm)"))
elif show == "Ten wettest days":
    t = r.sort_values(ascending=False).head(10).to_frame("Rain (mm)"); t.index = t.index.strftime("%Y-%m-%d"); display(t)
else:
    display(Markdown(f"Days with **{threshold_mm} mm or more**: **{(r >= threshold_mm).sum()}**"))

In [ ]:
#@title ▶ Check Q1 · Total rainfall in 2023 (mm)
total_mm = 0 #@param {type:"number"}
ans = rainfall.rain_mm.sum()
ok(f"Correct: {ans:,.1f} mm") if close(total_mm, ans) else no("Hint: add up the monthly totals in the explorer.")

In [ ]:
#@title ▶ Check Q2 · Number of rainy days (IMD definition, 2.5 mm or more)
rainy_days = 0 #@param {type:"number"}
ans = (rainfall.rain_mm >= 2.5).sum()
ok(f"Correct: {ans} rainy days") if rainy_days == ans else no("Hint: use 'Days at or above a threshold' with 2.5.")

In [ ]:
#@title ▶ Check Q3 · Wettest month
wettest_month = "choose" #@param ["choose", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ans = rainfall.rain_mm.resample("MS").sum().idxmax().strftime("%b")
ok(f"Correct: {ans}") if wettest_month == ans else no("Hint: look for the largest monthly total.")

In [ ]:
#@title ▶ Check Q4 · Percentage of the year's rain that fell on the 10 wettest days
percent = 0 #@param {type:"number"}
r = rainfall.rain_mm
ans = r.sort_values(ascending=False).head(10).sum() / r.sum() * 100
ok(f"Correct: {ans:.1f}%") if close(percent, ans, tol=0.05) else no("Hint: add the ten wettest days, divide by the year's total, multiply by 100.")

In [ ]:
#@title ▶ Check Q5 · Which season brought the most rain?
season = "choose" #@param ["choose", "Winter (Jan–Feb)", "Pre-monsoon (Mar–May)", "SW monsoon (Jun–Sep)", "NE monsoon (Oct–Dec)"]
ok("Correct: the southwest monsoon (June to September).") if season == "SW monsoon (Jun–Sep)" else no("Hint: use 'Seasonal totals' in the explorer.")